# Семинар 1. Знакомство с инструментами и базовый ETL-процесс

**Цель семинара:** Настроить рабочее окружение, освоить базовые возможности библиотек pandas и numpy, автоматизировать загрузку данных и реализовать сквозную функцию промышленной очистки данных.

### 🔧 Настройка рабочего окружения

Перед началом выполнения убедитесь, что ваше локальное окружение развернуто в соответствии со стандартами:

* Использован менеджер пакетов **UV** для управления зависимостями и версиями Python.


* В **VS Code** активны обязательные расширения `ms-python.python` (окружение Python) и `ms-toolsai.jupyter` (поддержка интерактивных ноутбуков).


* Подключен линтер `charliermarsh.ruff` для автоматического контроля качества и форматирования кода.



Сегодня мы выступим в роли инженера данных телеком-оператора. Нам предстоит обработать сырую выгрузку из CRM, локализовать типичные системные ошибки сбора данных и упаковать их исправление в чистую ETL-функцию.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

# Настройка принудительного отображения всех колонок датафрейма

pd.set_option('display.max_columns', None)


## 1. Автоматизированное получение датасета

Вместо ручного скачивания файлов мы настроим инструмент автоматического получения актуальной версии датасета **Telco Customer Churn** напрямую через API репозитория.


In [ ]:
print("Скачивание актуальной версии датасета...")
dataset_path = kagglehub.dataset_download("abbas829/telco-customer-churn-dataset", output_dir="./data/seminar_1")
csv_files = [f for f in os.listdir(dataset_path) if f.endswith('.csv')]

if not csv_files:
    raise FileNotFoundError("Критическая ошибка: целевой CSV-файл не обнаружен!")

full_csv_path = os.path.abspath(os.path.join(dataset_path, csv_files[0]))
print(f"Файл успешно инициализирован. Путь в файловой системе: {full_csv_path}")


## 🛠 Шаг 1 и 2: Чтение данных и первичный диагностический анализ

Преобразуем CSV-файл в объект `DataFrame` и проведем экспресс-диагностику типов данных, пропусков и ключевых статистических метрик для выявления аномалий.


In [ ]:
# Импорт сырых данных в рабочую область

df_raw = pd.read_csv(full_csv_path)

print("--- Первые 5 строк исходного датасета ---")
print(df_raw.head())

print("\n--- Технический паспорт таблицы (info) ---")
df_raw.info()

print("\n--- Описательная статистика числовых признаков (describe) ---")
print(df_raw.describe())


## 🛠 Шаг 3: Построение функции clean_and_normalise_dataframe

В реальной практике промышленной разработки аналитики избегают разрозненных модификаций колонок. Логика подготовки собирается в единую функцию трансформации датафрейма.

Мы реализуем функцию `clean_and_normalise_dataframe` для устранения следующих критических ошибок:

1. **Мусор в заголовках:** Удаление скрытых концевых пробелов и спецсимволов в названиях признаков.


2. **Невидимые пропуски / Текстовые заглушки:** Выявление скрытых пробелов в текстовых полях (`TotalCharges`) и замена их на системные `np.nan`.


3. **Числа в тексте:** Преобразование типов данных столбцов, неверно распознанных как `object`, в `float` или `int`.


4. **Явные пропуски:** Заполнение образовавшихся пустот нулями (поскольку отсутствие затрат наблюдается только у новых клиентов с нулевым жизненным циклом `tenure == 0`).


5. **Хаос во флагах:** Унификация кодирования категориальных бинарных признаков (перевод текстовых `Yes`/`No` в системные булевы флаги `True`/`False`).


6. **Полные дубликаты строк:** Очистка датасета от дублирующихся записей, возникших вследствие технических сбоев трансляции данных.


In [ ]:
def clean_and_normalise_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    # Изолируем исходный объект для предотвращения побочных эффектов модификации ссылок
    df_clean = df.copy()


    # 1. Исправление мусора в заголовках колонок 
    df_clean.columns = df_clean.columns.str.strip().str.replace(' ', '_')

    # 2. Обработка невидимых пропусков в TotalCharges (строки из пробелов) 
    df_clean['TotalCharges'] = df_clean['TotalCharges'].replace(r'^\s*$', np.nan, regex=True)

    # 3. Трансформация чисел, сохраненных как текст, в корректный тип данных float 
    df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

    # 4. Обработка явных и скрытых пропусков (заполнение нулями логически обоснованных ячеек) 
    df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(0)

    # 5. Устранение хаоса во флагах: преобразование текстовых Yes/No в единый булев формат 
    flag_mapping = {'Yes': True, 'No': False}
    binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
    for col in binary_cols:
        if col in df_clean.columns:
            # Преобразуем только явные значения, сохраняя структуру
            df_clean[col] = df_clean[col].map(flag_mapping)

    # 6. Удаление полных дубликатов строк 
    df_clean = df_clean.drop_duplicates()

    return df_clean



# Применение разработанного пайплайна к сырому датасету

df = clean_and_normalise_dataframe(df_raw)

print(f"Размер исходных данных: {df_raw.shape}")
print(f"Размер очищенных нормализованных данных: {df.shape}")
print(f"Количество пропусков в TotalCharges после обработки: {df['TotalCharges'].isnull().sum()}")


## 🛠 Шаг 4: Анализ распределения и визуализация

Для оценки дисбаланса классов в задаче прогнозирования оттока построим столбчатую диаграмму по распределению преобразованного целевого признака `Churn`.


In [ ]:
# Подсчет частот распределения классов оттока

churn_counts = df['Churn'].value_counts()

# Отрисовка результатов аналитики

plt.figure(figsize=(7, 4))
plt.bar(
churn_counts.index.map({True: 'Churn (Ушли)', False: 'Active (Остались)'}),
churn_counts.values,
color=['#4CAF50', '#F44336'],
alpha=0.85
)
plt.title('Распределение оттока клиентов (Target: Churn)', fontsize=12, fontweight='bold')
plt.xlabel('Текущий статус контракта', fontsize=10)
plt.ylabel('Количество клиентов (абс.)', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


## 🛠 Автоматизированная проверка качества (Autocheck)

Запустим валидационный скрипт, проверяющий соответствие обработанного датафрейма архитектурным требованиям новой учебной программы.


In [ ]:
def run_autocheck():
    print("🚀 Тестирование обработанного датафрейма...\n" + "-"*40)
    validation_status = True
    
    
    # Валидация структуры таблицы
    if not isinstance(df, pd.DataFrame):
        print("❌ Ошибка: Результирующий объект не является валидным DataFrame.")
        validation_status = False
        
    # Валидация чистоты заголовков
    if any(df.columns.str.contains(r'^\s|\s$')):
        print("❌ Ошибка: В именах столбцов обнаружены концевые пробелы.")
        validation_status = False
    else:
        print("✅ Заголовки колонок успешно нормализованы.")
        
    # Валидация типов данных и заполнения NaN
    if not pd.api.types.is_numeric_dtype(df['TotalCharges']):
        print("❌ Ошибка: Признак TotalCharges имеет некорректный нечисловой тип.")
        validation_status = False
    elif df['TotalCharges'].isnull().sum() > 0:
        print("❌ Ошибка: В столбце TotalCharges не устранены пропущенные значения.")
        validation_status = False
    else:
        print("✅ Типизация и пропуски в TotalCharges исправлены.")
        
    # Валидация дубликатов
    if df.duplicated().sum() > 0:
        print("❌ Ошибка: Из датасета не удалены идентичные строки-дубликаты.")
        validation_status = False
    else:
        print("✅ Дубликаты в таблице отсутствуют.")
        
    # Валидация нормализации логических признаков
    if df['Churn'].dtype != bool:
        print("❌ Ошибка: Целевой признак Churn не приведен к булевому типу.")
        validation_status = False
    else:
        print("✅ Кодирование логических флагов стандартизировано.")
    
    print("-" * 40)
    if validation_status:
        print("🎉 ПОЗДРАВЛЯЕМ! Разработанный ETL-пайплайн полностью соответствует требованиям программы.")
    else:
        print("⚠️ Обнаружены технические дефекты. Проверьте реализацию функции clean_and_normalise_dataframe.")



run_autocheck()